# 🛰️ Sentinel-1 Download Pipeline Notebook: Overview
## **Automated Spatial Querying → Pass-Aware Filtering → Multithreaded .SAFE Retrieval**

This notebook serves as the **core data acquisition engine** for the Sentinel-1 download workflow. It takes a Region of Interest (ROI) and a time window, queries the Copernicus Data Space Ecosystem (CDSE) OData catalogue for Sentinel-1 SAR products, applies SAR-specific filters (sensor mode, product type, polarisation, orbit direction, relative orbit), previews the selection on a map with a volume estimate, and then downloads the products as `.zip` archives containing the native `.SAFE` structure.

---

### 🚀 Core Functional Areas
* **Spatial Validation & Verification**: Interactive `folium` mapping verifies the ROI before querying, and a second map shows the selected Sentinel-1 footprints (ascending = blue, descending = orange) against the ROI before anything is downloaded.
* **SAR-Aware Data Orchestration**: Paginated catalogue queries (no 1,000-product truncation), COG-duplicate removal, reprocessing de-duplication and a **pass-level coverage filter** that unions consecutive slices of the same datatake, so an ROI straddling two slices is not discarded.
* **Multithreaded Retrieval & Resume**: Parallel downloads with automatic token refresh, `.part` files with size verification, and a per-product CSV log that is written as each download completes so interrupted runs resume cleanly.

---

### 🔁 Sentinel-1 Selection Logic
| Aspect | Behaviour |
| :--- | :--- |
| **Cloud filter** | **None** — SAR images through cloud, and Sentinel-1 products carry no `cloudCover` attribute. |
| **Product selection** | `sensor_mode` (`IW`, `EW`…) + `product_type` (`GRDH`, `SLC`…) + `polarisation` (`DV`, `DH`…). |
| **Version selection** | Latest `PublicationDate` per identical sensing window (reprocessed versions differ only in the CRC). |
| **Spatial unit** | Along-track slice → ROI coverage evaluated per **pass** (all slices sharing a datatake ID). |
| **Acquisition geometry** | `orbit_direction`, `relative_orbit` and `missions` restrict passes to a consistent viewing geometry. |
| **Duplicates** | `exclude_cog` drops the `*_COG.SAFE` repackaged copies of GRD products. |

---

### 📋 Execution Requirements & Troubleshooting
* **CDSE Credentials**: Requires an active Copernicus Data Space Ecosystem account to generate OAuth2 bearer tokens.
* **Storage Allocation**: IW GRDH archives are ~1–1.7 GB per slice; IW SLC archives are ~4–8 GB. Always run the **preview cell** first — it reports the total download volume.
* **Nothing selected?** Lower `min_roi_coverage_frac`, set `enable_pass_coverage_filter=False`, or relax `polarisation` / `orbit_direction` / `relative_orbit`.

---

### 👤 Author: Julian Manning
♻️ Last updated: **October 2026**  
📝 Project: **Sentinel-1 Download Workflow**  
📧 [julian.manning@outlook.com](mailto:julian.manning@outlook.com)  
🔗 Connect on [LinkedIn](https://www.linkedin.com/in/julian-manning/)  

<p style="text-align:left">
    <a href="https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-1" target="_blank">
        <img src=https://www.esa.int/var/esa/storage/images/applications/observing_the_earth/copernicus/sentinel-1/19729210-29-eng-GB/Sentinel-1_pillars.png width="400" alt="Graphic">
    </a>
</p>

## ⚙️ Workspace Configuration & Dependency Imports
> ### **Standard Library Initialisation → Custom Module Integration**

This configures the system path to recognise the local `Tools` directory and imports the core orchestration functions from the **Sentinel-1 Download Pipeline Workflow** utility.

---

### 📂 Orchestration Components
| Function | Responsibility |
| :--- | :--- |
| **`visualise_roi_on_map`** | Generates an interactive Folium map of the Region of Interest (ROI) for spatial verification. |
| **`get_roi_wkt_from_vector`** | Reads a vector dataset, reprojects it to EPSG:4326 and returns a WKT bounding box (or simplified polygon). |
| **`search_s1_products`** | Runs the full query → filter → de-duplicate → pass-coverage chain **without downloading** (preview). |
| **`summarise_s1_products`** | Tabulates the selection (date, mission, direction, track, polarisation, size, coverage) and prints total volume. |
| **`visualise_s1_footprints`** | Maps selected footprints against the ROI, coloured by orbit direction. |
| **`download_copernicus_s1_data`** | Orchestrates the end-to-end pipeline for querying, filtering and downloading Sentinel-1 data. |
| **`resume_copernicus_s1_data_download`** | Re-runs the pipeline, skipping product UUIDs already recorded in the CSV log. |
| **`download_timer`** | Context manager that measures and logs the execution runtime of a code block. |

In [1]:
# Core
import sys
import getpass
from pathlib import Path

# Geospatial
import geopandas as gpd

# Ensure the 'Tools' directory is in the path for custom module discovery
sys.path.insert(0, "Tools")

from s1_download_helper_utility import (
    visualise_roi_on_map,
    get_roi_wkt_from_vector,
    search_s1_products,
    summarise_s1_products,
    visualise_s1_footprints,
    download_copernicus_s1_data,
    resume_copernicus_s1_data_download,
    download_timer
)

## 🗺️ Region of Interest (ROI) Initialisation & Spatial Validation
> ### **Vector Data Ingestion → Interactive Cartographic Verification**

This section defines the **Region of Interest (ROI)** and performs a quick **spatial sanity check** to confirm geographic alignment before any catalogue query is run.

---

### 📂 Logic & Orchestration Components
| Component | Responsibility | Analytical Objective |
| :--- | :--- | :--- |
| **`ROI_SHP_PATH`** | **Universal Path Resolution** | Utilises `Path` objects for cross-platform compatibility. Any CRS is accepted — it is reprojected to EPSG:4326 automatically. |
| **`ROI_GDF`** | **Vector Data Ingestion** | Reads the shapefile (or GeoPackage / GeoJSON) into a GeoDataFrame. |
| **`visualise_roi_on_map`** | **Spatial Sanity Check** | Renders an interactive map to verify ROI alignment against global basemaps. |
| **`get_roi_wkt_from_vector`** | **API Geometry** | Returns the ROI bounding box as WKT (default). Use `use_bbox=False` to query with the actual (simplified) polygon instead. |

---

### 📋 Optional Tips
* **Basemap Customisation**: `"OpenStreetMap"`, `"CartoDB positron"`, `"CartoDB voyager"`, `"Esri.WorldImagery"`, `"CartoDB dark_matter"`.
* **Geometric Aggregation**: Set `use_dissolve=True` if your ROI consists of multiple features.
* **Bounding box vs polygon**: For long, thin or diagonal ROIs (coastlines, rivers), `use_bbox=False` avoids pulling in passes that only touch the empty corners of the bounding box.
* **Reporting & Export**: Use `save_path=` to export the map as a standalone HTML file.

In [3]:
# 1. Define your path
ROI_SHP_PATH = Path(r"DATA\SHP\Strait_of_Hormuz_Extent_EPSG_32640.shp")
# 2. Visualise it (optional sanity check)
ROI_GDF = gpd.read_file(ROI_SHP_PATH)
display(visualise_roi_on_map(ROI_GDF, tiles="Esri.WorldImagery"))   # or "Esri.WorldImagery", "OpenStreetMap"
# 3. Extract the WKT (bounding box by default; use_bbox=False for the actual polygon)
roi_wkt_string = get_roi_wkt_from_vector(ROI_SHP_PATH, use_bbox=True)
print(roi_wkt_string)

POLYGON ((58.54210993022871 23.52216925589167, 58.54210993022871 28.03584180864551, 53.765616013707174 28.03584180864551, 53.765616013707174 23.52216925589167, 58.54210993022871 23.52216925589167))


## 🔐 Authentication & Credential Management
> ### **Secure Runtime Input → CDSE Keycloak Identity Verification**

This handles the secure, hidden ingestion of user credentials at runtime, required to request an OAuth2 bearer token from the CDSE Keycloak service. Credentials are only used for **downloading** — the preview step queries the public catalogue anonymously.

> 🌐 **Prerequisite**: Sign into your Copernicus account as you normally do [here](https://dataspace.copernicus.eu/).
> ⚠️ **Note**: If you do not have an active CDSE account, you must register a new profile before proceeding.

---

### 📂 Authentication Parameters
| Variable | Responsibility |
| :--- | :--- |
| **`copernicus_user`** | Captures your registered CDSE email/username via a secure text input prompt. |
| **`copernicus_password`** | Masked runtime input that prevents cleartext exposure in logs. |

In [3]:
# copernicus User email
copernicus_user = getpass.getpass("Enter your email address")
# copernicus User Password
copernicus_password = getpass.getpass("Enter your password")

Enter your email address ········
Enter your password ········


## 🚀 Pipeline Parameter Configuration
> ### **Pipeline Parameterisation → SAR Product Selection**

This stage configures the temporal, product and acquisition-geometry constraints for the dataset. The same `pipeline_config` dictionary drives the **preview**, **download** and **resume** cells.

---

### 📂 Configuration Parameters
| Parameter | Default/Value | Responsibility |
| :--- | :--- | :--- |
| **`time_begin` / `time_end`** | `'2026-08-01'` / `'2026-08-31'` | Sensing window (`YYYY-MM-DD`). **Both dates are inclusive.** |
| **`sensor_mode`** | `"IW"` | Acquisition mode: `IW` (Interferometric Wide, default over land/coast), `EW` (Extra Wide, polar/maritime), `S1`–`S6` (Stripmap), `WV` (Wave). |
| **`product_type`** | `"GRDH"` | `GRDH` (GRD high-res, backscatter), `GRDM`, `GRDF`, `SLC` (phase, InSAR), `RAW`, `OCN`. |
| **`polarisation`** | `"DV"` | `"DV"` = VV+VH, `"DH"` = HH+HV, `"SV"` = VV only, `"SH"` = HH only, or `None` for all. A list is also accepted, e.g. `["DV", "SV"]`. |
| **`orbit_direction`** | `None` | `"ASCENDING"`, `"DESCENDING"`, or `None` for both. |
| **`relative_orbit`** | `None` | Track number(s), e.g. `9` or `[9, 111]`. Read the values from the preview table first. |
| **`missions`** | `None` | Restrict platforms, e.g. `["S1A", "S1C"]`. `None` keeps all. |
| **`exclude_cog`** | `True` | Drops the `*_COG.SAFE` repackaged duplicates CDSE publishes for GRD products. |
| **`download_latest_only`** | `True` | Keeps only the most recently published version of each identical sensing window. |
| **`enable_pass_coverage_filter`** | `True` | Drops passes that cover less than `min_roi_coverage_frac` of the ROI. |
| **`min_roi_coverage_frac`** | `0.70` | Minimum fraction of the ROI that a pass (all its slices combined) must cover. |
| **`save_folder`** | `Path(r"DATA")` | Target directory for `.zip` archives and the CSV log. |
| **`max_parallel_downloads`** | `2` | Concurrent downloads via `ThreadPoolExecutor`. |
| **`time_out`** | `2400` | HTTP timeout (seconds) before dropping dead connections. |

---

> ⚠️ **Performance & Stability Note**: CDSE limits the number of concurrent downloads per account. `2`–`4` workers is a sensible range; if you encounter `403 Forbidden` or `429 Too Many Requests`, set `max_parallel_downloads` to `1`.

---

### 🗺️ Sentinel-1 Naming Convention
The pipeline parses product filenames based on the standard Sentinel-1 SAFE format:
`MMM_BB_TTTR_LFPP_YYYYMMDDTHHMMSS_YYYYMMDDTHHMMSS_OOOOOO_DDDDDD_CCCC.SAFE`

* **`MMM`**: Mission identifier (`S1A`, `S1B` — inactive since Dec 2021, `S1C`, `S1D`).
* **`BB`**: Sensor mode / beam (`IW`, `EW`, `WV`, `S1`–`S6`).
* **`TTTR`**: Product type + resolution (`GRDH`, `GRDM`, `SLC_`, `RAW_`, `OCN_`).
* **`L`**: Processing level (`1` for GRD/SLC, `2` for OCN, `0` for RAW).
* **`F`**: Product class (`S` = Standard, `A` = Annotation).
* **`PP`**: Polarisation (`DV` = VV+VH, `DH` = HH+HV, `SV` = VV, `SH` = HH).
* **`YYYYMMDDTHHMMSS` ×2**: Sensing start and stop time.
* **`OOOOOO`**: Absolute orbit number.
* **`DDDDDD`**: **Mission datatake ID** — shared by all consecutive slices of one pass; used for pass-level coverage grouping.
* **`CCCC`**: Product unique identifier (CRC) — differs between reprocessed versions.
* **`_COG`** *(optional)*: Cloud-Optimised GeoTIFF repackaging of the same product.

> 📌 **Version Selection Mechanics**: When multiple products share the identical mission, mode, type, polarisation and sensing start/stop times, the most recently published one is retained:
> * `..._20260810T190026_20260810T190051_060521_078A2B_CCCC.SAFE` (published 2026-09-01 ✅ Kept)
> * `..._20260810T190026_20260810T190051_060521_078A2B_BBBB.SAFE` (published 2026-08-12 ❌ Dropped)

> 📌 **Pass Coverage Mechanics**: Sentinel-1 data is cut into ~25 s slices along track. If your ROI straddles two slices of the same pass (each covering ~50 %), their footprints are unioned before testing coverage, so both are kept (100 % ✅) — whereas a per-slice threshold would have dropped both.

In [4]:
year = '2026'
# Set your core variables
time_begin = f"{year}-10-01"
time_end = f"{year}-10-20"
save_folder = Path(fr"DATA\S1\{year}")

# Bundle ALL shared parameters into a single dictionary
pipeline_config = {
    "data_collection": "SENTINEL-1",
    "ROI": roi_wkt_string,
    "time_begin": time_begin,                 # Format: YYYY-MM-DD (inclusive)
    "time_end": time_end,                     # Format: YYYY-MM-DD (inclusive)
    "copernicus_user": copernicus_user,
    "copernicus_password": copernicus_password,
    "sensor_mode": "IW",                      # "IW", "EW", "S1"-"S6", "WV"
    "product_type": "GRDH",                   # "GRDH", "GRDM", "SLC", "OCN", "RAW"
    "polarisation": "DV",                     # "DV" (VV+VH), "DH", "SV", "SH" or None
    "orbit_direction": None,                  # "ASCENDING", "DESCENDING" or None
    "relative_orbit": None,                   # e.g. 9 or [9, 111]; None = all tracks
    "missions": None,                         # e.g. ["S1A", "S1C"]; None = all
    "exclude_cog": True,                      # Drop *_COG.SAFE duplicates
    "save_folder": str(save_folder),          # Converts Path object to string for the function
    "time_out": 2400,                         # Timeout in seconds
    "download_latest_only": True,             # Keep the latest published version per scene
    "enable_pass_coverage_filter": True,      # Drop passes that barely touch the ROI
    "min_roi_coverage_frac": 0.10,            # Pass must cover >= 70% of the ROI
    "max_parallel_downloads": 2,              # Set >1 for faster threaded downloads
}

# Dynamically construct the log file name based on the dates
t1 = time_begin.replace("-", "_")
t2 = time_end.replace("-", "_")
existing_log_file = save_folder / f"log_{t1}_to_{t2}.csv"

print(f"Targeting log file: {existing_log_file}")

Targeting log file: DATA\S1\2026\log_2026_10_01_to_2026_10_20.csv


## 🔎 Selection Preview (No Download)
> ### **Catalogue Query → Filter Audit → Footprint Map & Volume Estimate**

This cell runs the complete selection chain — query, Sentinel-1 filters, reprocessing de-duplication and pass coverage — **without downloading anything**. Use it to check what you are about to pull, how large it is, and which relative orbits cover your ROI before committing bandwidth and disk.

---

### 📂 Preview Outputs
| Output | Responsibility |
| :--- | :--- |
| **Console audit** | Prints the query URL, how many products each filter removed and a ✓/✗ coverage line per pass. |
| **`summary_df`** | One row per product: date, mission, direction, relative orbit, polarisation, size (GB), slice and pass coverage. |
| **Footprint map** | Ascending passes in blue, descending in orange, ROI in red. Hover for product names. |

---

> 💡 **Tip**: For time-series or change-detection work, choose a single `relative_orbit` (and therefore one `orbit_direction`) so every scene shares the same viewing geometry. The `Rel. orbit` column shows which tracks are available. `summary_df["Rel. orbit"].value_counts()` gives a quick count per track.

In [5]:
with download_timer("Sentinel-1 Selection Preview"):
    preview_products = search_s1_products(**pipeline_config)

summary_df = summarise_s1_products(preview_products)
display(summary_df)
display(visualise_s1_footprints(preview_products, roi_wkt_string))


--- Starting Sentinel-1 Selection Preview ---
Start time: 2026-10-23 16:40:11

Starting query to Copernicus Data Space (Sentinel-1)...
Query URL:
https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name%20eq%20'SENTINEL-1'%20and%20OData.CSC.Intersects(area=geography'SRID=4326;POLYGON%20((58.54210993022871%2023.52216925589167,%2058.54210993022871%2028.03584180864551,%2053.765616013707174%2028.03584180864551,%2053.765616013707174%2023.52216925589167,%2058.54210993022871%2023.52216925589167))')%20and%20ContentDate/Start%20ge%202026-10-01T00:00:00.000Z%20and%20ContentDate/Start%20lt%202026-10-21T00:00:00.000Z%20and%20contains(Name,'_IW_GRDH_')&$orderby=ContentDate/Start%20asc&$expand=Attributes&$count=True&$top=500
Catalogue reports 176 matching products.
Retrieved 176 products in 1 page(s).
Raw product count: 176

Applying Sentinel-1 filters...
  COG duplicates: dropped 88
Products after Sentinel-1 filters: 88 (from 176)
Reprocessed duplicates removed: 0 (kept 

,Date,Mission,Mode,Type,Direction,Rel. orbit,Pol,Size (GB),ROI cover %,Pass cover %,Name
0,2026-10-01,S1D,IW,GRDH,ASCENDING,57,DV (VV+VH),1.74,14.47,53.87,S1D_IW_GRDH_1SDV_20261001T141539_20261001T1416...
1,2026-10-01,S1D,IW,GRDH,ASCENDING,57,DV (VV+VH),1.74,18.32,53.87,S1D_IW_GRDH_1SDV_20261001T141604_20261001T1416...
2,2026-10-01,S1D,IW,GRDH,ASCENDING,57,DV (VV+VH),1.74,18.32,53.87,S1D_IW_GRDH_1SDV_20261001T141629_20261001T1416...
3,2026-10-01,S1D,IW,GRDH,ASCENDING,57,DV (VV+VH),1.74,2.75,53.87,S1D_IW_GRDH_1SDV_20261001T141654_20261001T1417...
4,2026-10-02,S1D,IW,GRDH,DESCENDING,64,DV (VV+VH),1.73,4.10,17.35,S1D_IW_GRDH_1SDV_20261002T022215_20261002T0222...
...,...,...,...,...,...,...,...,...,...,...,...
65,2026-10-20,S1C,IW,GRDH,DESCENDING,64,DV (VV+VH),1.33,1.75,15.39,S1C_IW_GRDH_1SDV_20261020T022315_20261020T0223...
66,2026-10-20,S1D,IW,GRDH,ASCENDING,159,DV (VV+VH),1.73,0.89,14.92,S1D_IW_GRDH_1SDV_20261020T140718_20261020T1407...
67,2026-10-20,S1D,IW,GRDH,ASCENDING,159,DV (VV+VH),1.73,3.82,14.92,S1D_IW_GRDH_1SDV_20261020T140743_20261020T1408...
68,2026-10-20,S1D,IW,GRDH,ASCENDING,159,DV (VV+VH),1.73,6.08,14.92,S1D_IW_GRDH_1SDV_20261020T140808_20261020T1408...


## 🚀 Pipeline Execution
> ### **Threaded Orchestration & Profiling**

Runs the end-to-end pipeline (re-running the same selection as the preview) and downloads each product as `<product_name>.zip` into `save_folder`, wrapped in an execution profile timer.

---

### 📋 Download Behaviour
* **Atomic writes**: Data streams to `*.zip.part` and is only renamed to `.zip` once the byte count matches the catalogue's `ContentLength`.
* **Already-present files**: A `.zip` that already exists with the expected size is skipped, not re-downloaded.
* **Live logging**: Each successful UUID is appended to the CSV log as soon as it completes — an interrupted run loses at most the files that were in flight.
* **Token management**: The Keycloak token is refreshed automatically every ~8 minutes and on any `401` response.

In [6]:
# Run the end-to-end download pipeline and track the time
with download_timer("Sentinel-1 Download Pipeline"):
    try:
        # The ** operator unpacks all parameters from the dictionary dynamically
        downloaded_files = download_copernicus_s1_data(**pipeline_config)

        print(f"\nSuccessfully downloaded {len(downloaded_files)} files to: {save_folder}")

    except Exception as e:
        print(f"\x1b[91mPipeline failed during execution: {e}\x1b[0m")


--- Starting Sentinel-1 Download Pipeline ---
Start time: 2026-10-23 16:42:28

Starting query to Copernicus Data Space (Sentinel-1)...
Query URL:
https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name%20eq%20'SENTINEL-1'%20and%20OData.CSC.Intersects(area=geography'SRID=4326;POLYGON%20((58.54210993022871%2023.52216925589167,%2058.54210993022871%2028.03584180864551,%2053.765616013707174%2028.03584180864551,%2053.765616013707174%2023.52216925589167,%2058.54210993022871%2023.52216925589167))')%20and%20ContentDate/Start%20ge%202026-10-01T00:00:00.000Z%20and%20ContentDate/Start%20lt%202026-10-21T00:00:00.000Z%20and%20contains(Name,'_IW_GRDH_')&$orderby=ContentDate/Start%20asc&$expand=Attributes&$count=True&$top=500
Catalogue reports 176 matching products.
Retrieved 176 products in 1 page(s).
Raw product count: 176

Applying Sentinel-1 filters...
  COG duplicates: dropped 88
Products after Sentinel-1 filters: 88 (from 176)
Reprocessed duplicates removed: 0 (kept 

  0%|          | 0/70 [00:00<?, ?it/s]

  S1D_IW_GRDH_1SDV_20261001T141604_2026100…  25.2% 0.44 GB [00:02:07]

  1%|▏         | 1/70 [02:13<2:33:06, 133.14s/it]


Downloaded: S1D_IW_GRDH_1SDV_20261001T141539_20261001T141604_004823_0090E6_DA27.SAFE (1.74 GB) in 00:02:13
  S1D_IW_GRDH_1SDV_20261001T141604_2026100…  97.9% 1.70 GB [00:06:49]

...


Downloaded: S1D_IW_GRDH_1SDV_20261020T140743_20261020T140808_005100_009A78_8BCC.SAFE (1.73 GB) in 00:04:08
  S1D_IW_GRDH_1SDV_20261020T140808_2026102…  97.4% 1.69 GB [00:10:26]

 99%|█████████▊| 69/70 [2:59:59<04:33, 273.45s/it]


Downloaded: S1D_IW_GRDH_1SDV_20261020T140833_20261020T140858_005100_009A78_C9F6.SAFE (1.73 GB) in 00:07:23
  S1D_IW_GRDH_1SDV_20261020T140808_2026102…  99.2% 1.72 GB [00:10:47]

100%|██████████| 70/70 [3:00:22<00:00, 154.60s/it]


Downloaded: S1D_IW_GRDH_1SDV_20261020T140808_20261020T140833_005100_009A78_2F02.SAFE (1.73 GB) in 00:10:51

Log file: DATA\S1\2026\log_2026_10_01_to_2026_10_20.csv

Successfully downloaded 70 files to: DATA\S1\2026

--- Sentinel-1 Download Pipeline Complete ---
Start time: 2026-10-23 16:42:28
End time:   2026-10-23 19:42:59
Elapsed time: 3h 0m 31s


## 🔄 Pipeline Recovery & Log Parsing
> ### **CSV Ingestion → Targeted Retrieval**

This stage ingests the existing CSV log of successfully downloaded Copernicus **UUIDs** and re-runs the query, isolating only the products that are still missing. If a large batch is interrupted by network instability, this resumes the queue without re-downloading anything already acquired.

---

### 📂 Recovery Parameters
| Parameter | Default/Value | Responsibility |
| :--- | :--- | :--- |
| **`log_csv_path`** | `existing_log_file` | Path to the CSV containing the UUIDs of previously downloaded products. |
| ***(Standard Parameters)*** | *(Inherited)* | Uses the exact same temporal, spatial, product and authentication parameters as the standard pipeline. |

---

> 📌 **Queue Filtering Mechanics**: The active OData query results are cross-referenced against the CSV log:
> * `1dda1d9b-2d4c-4b0f-b786-b04bbeb18f80` (Found in CSV ❌ Skipped)
> * `830e92c0-3aca-4ccb-b867-573c809f200f` (Not in CSV ✅ Flagged for Download)
>
> 🛡️ **Data Integrity Assurance**: The log is only ever **appended to** (`"a"` mode) — progress is never overwritten. Any partially written `*.zip.part` files from the interrupted run are simply overwritten by the fresh download.

In [ ]:
# Run the RESUME pipeline
with download_timer("Sentinel-1 Download Resume Pipeline"):
    try:
        downloaded_files = resume_copernicus_s1_data_download(
            **pipeline_config,                    # Unpacks all the identical settings from above
            log_csv_path=str(existing_log_file)   # Passes the dynamically generated log file path
        )

        print(f"\nSuccessfully downloaded {len(downloaded_files)} new files to: {save_folder}")

    except Exception as e:
        print(f"\x1b[91mPipeline failed during execution: {e}\x1b[0m")